# RootSignal GPU training evidence
Reproduce the published LoRA training manifest on a CUDA GPU. Select a GPU runtime before running all cells.

In [ ]:
import os, platform, subprocess, sys, time
print(subprocess.check_output('nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader', shell=True, text=True))
repo = '/content/rootsignal-bench'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', 'https://github.com/medhavee-upadhyaya/rootsignal-bench.git', repo], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo + '[train]'], check=True)
# Colab may include an old optional torchao build that is incompatible with current PEFT.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

In [ ]:
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'training.build_dataset', '--fixtures', 'fixtures/incidents', '--output-dir', '/content/rootsignal-training-data', '--eval-fraction', '0.2', '--seed', '17'], check=True)
subprocess.run([sys.executable, '-m', 'training.validate_artifacts', '--dataset-manifest', '/content/rootsignal-training-data/dataset_manifest.json'], check=True)

In [ ]:
started = time.time()
subprocess.run([sys.executable, '-m', 'training.train_lora', '--dataset-manifest', '/content/rootsignal-training-data/dataset_manifest.json', '--model', 'Qwen/Qwen2.5-0.5B-Instruct', '--output', '/content/rootsignal-tool-selector-lora', '--seed', '17', '--epochs', '3'], check=True)
print(f'wall_seconds={time.time() - started:.3f}')
subprocess.run([sys.executable, '-m', 'training.validate_artifacts', '--training-manifest', '/content/rootsignal-tool-selector-lora/rootsignal_manifest.json'], check=True)

In [ ]:
from pathlib import Path
print(Path('/content/rootsignal-tool-selector-lora/rootsignal_manifest.json').read_text())